In [ ]:
%matplotlib inline

from dataclasses import dataclass
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Markdown
except ImportError:
    display = None
    Markdown = None
from scipy.optimize import curve_fit, linear_sum_assignment, root
from scipy.signal import find_peaks, peak_widths
from scipy.special import spherical_jn, spherical_yn


def _section(title):
    if display is not None and Markdown is not None:
        display(Markdown(f"## {title}"))
    else:
        print(f"\n=== {title} ===")


def _display_table(df, title=None, max_rows=None):
    if title:
        _section(title)
    shown = df if max_rows is None else df.head(max_rows)
    if display is not None:
        display(shown)
    else:
        print(shown.to_string(index=False))


def _finish_figure(fig, save=None, show=True, close=True):
    if save:
        fig.savefig(save, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    if close:
        plt.close(fig)


def _asarray(x):
    a = np.asarray(x)
    return a.astype(complex if np.iscomplexobj(a) else float)


def J(n, x):
    x = _asarray(x)
    if n >= 0:
        return spherical_jn(n, x)
    m = -n - 1
    return (-1) ** (m + 1) * spherical_yn(m, x)


def Y(n, x):
    x = _asarray(x)
    if n >= 0:
        return spherical_yn(n, x)
    m = -n - 1
    return (-1) ** m * spherical_jn(m, x)


def dJ(n, x):
    x = _asarray(x)
    if n >= 0:
        return spherical_jn(n, x, derivative=True)
    m = -n - 1
    return (-1) ** (m + 1) * spherical_yn(m, x, derivative=True)


def dY(n, x):
    x = _asarray(x)
    if n >= 0:
        return spherical_yn(n, x, derivative=True)
    m = -n - 1
    return (-1) ** m * spherical_jn(m, x, derivative=True)


def H(n, x):
    return J(n, x) + 1j * Y(n, x)


def dH(n, x):
    return dJ(n, x) + 1j * dY(n, x)


@dataclass(frozen=True)
class SphereStudyConfig:
    rhoM: float = 1024.0
    cM: float = 1522.0
    xM_min: float = 1.0e-4
    xM_max: float = 1.0
    n_x: int = 1800
    lmax: int = 10
    prominence_log10: float = 0.35
    min_distance_x: float = 0.0
    width_rel_height: float = 0.5

    def x_grid(self):
        if self.xM_min <= 0 or self.xM_max <= self.xM_min or self.n_x < 10:
            raise ValueError("A grade de xM é inválida.")
        return np.geomspace(self.xM_min, self.xM_max, self.n_x)


def validate_material(rhoM, rhoA, cM, cA, lmax):
    vals = {"rhoM": rhoM, "rhoA": rhoA, "cM": cM, "cA": cA}
    for name, value in vals.items():
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f"{name} deve ser positivo e finito.")
    if int(lmax) != lmax or lmax < 0:
        raise ValueError("lmax deve ser um inteiro não negativo.")


def sphere_reference_x(rho_ratio=None, c_ratio=None, impedance_ratio=None, f_mix=None):
    if rho_ratio is not None and c_ratio is not None:
        rho_ratio = np.asarray(rho_ratio, dtype=float)
        c_ratio = np.asarray(c_ratio, dtype=float)
        if np.any(rho_ratio <= 0) or np.any(c_ratio <= 0):
            raise ValueError("rho_ratio e c_ratio devem ser positivos.")
        return np.sqrt(3.0 * rho_ratio * c_ratio ** 2)
    if impedance_ratio is None or f_mix is None:
        raise ValueError("Informe (rho_ratio,c_ratio) ou (impedance_ratio,f_mix).")
    impedance_ratio = np.asarray(impedance_ratio, dtype=float)
    f_mix = np.asarray(f_mix, dtype=float)
    if np.any(impedance_ratio <= 0) or np.any((f_mix < 0) | (f_mix > 1)):
        raise ValueError("impedance_ratio deve ser positivo e 0 <= f_mix <= 1.")
    return np.sqrt(3.0 * impedance_ratio ** (2.0 - f_mix))


def ratios_from_impedance(impedance_ratio, f_mix):
    zeta = np.asarray(impedance_ratio, dtype=float)
    if np.any(zeta <= 0) or not 0 <= f_mix <= 1:
        raise ValueError("impedance_ratio deve ser positivo e 0 <= f_mix <= 1.")
    return zeta ** f_mix, zeta ** (1.0 - f_mix)


def sphere_coefficients(xM_vals, rhoM, rhoA, cM, cA, lmax):
    validate_material(rhoM, rhoA, cM, cA, lmax)
    xM = _asarray(xM_vals)
    if np.any(np.real(xM) <= 0) and not np.iscomplexobj(xM):
        raise ValueError("Todos os valores reais de xM devem ser positivos.")
    xA = xM * cM / cA
    rho_ratio = rhoM / rhoA
    impedance_ratio = rhoM * cM / (rhoA * cA)
    shape = (lmax + 1,) + xM.shape
    b = np.zeros(shape, dtype=complex)
    s = np.zeros(shape, dtype=complex)
    denominator = np.zeros(shape, dtype=complex)
    for ell in range(lmax + 1):
        jA, djA = J(ell, xA), dJ(ell, xA)
        jM, djM = J(ell, xM), dJ(ell, xM)
        hM, dhM = H(ell, xM), dH(ell, xM)
        den = jA * dhM - impedance_ratio * djA * hM
        num_s = djM * jA - impedance_ratio * jM * djA
        denominator[ell] = den
        valid = np.abs(den) > 1.0e-300
        b[ell][valid] = 1j * rho_ratio / (xM[valid] ** 2 * den[valid])
        s[ell][valid] = -num_s[valid] / den[valid]
    return {"xM": xM, "xA": xA, "b": b, "s": s, "denominator": denominator}


def sphere_energy_parts_vs_xM(xM_vals, rhoM, rhoA, cM, cA, lmax):
    xM = np.asarray(xM_vals, dtype=float)
    if np.any(xM <= 0):
        raise ValueError("Todos os valores de xM devem ser positivos.")
    coeff = sphere_coefficients(xM, rhoM, rhoA, cM, cA, lmax)
    xA, b = np.real(coeff["xA"]), coeff["b"]
    modal_WP = np.zeros((lmax + 1, xM.size), dtype=float)
    modal_WK = np.zeros_like(modal_WP)
    for ell in range(lmax + 1):
        jA, djA = J(ell, xA), dJ(ell, xA)
        radial = 0.5 * xA ** 3 * (jA ** 2 - J(ell - 1, xA) * J(ell + 1, xA))
        boundary = xA ** 2 * jA * djA
        weight = (2 * ell + 1) * np.abs(b[ell]) ** 2
        modal_WP[ell] = np.maximum(np.real(weight * radial), 0.0)
        modal_WK[ell] = np.maximum(np.real(weight * (radial + boundary)), 0.0)
    WP = np.sum(modal_WP, axis=0)
    WK = np.sum(modal_WK, axis=0)
    total = np.where(np.isfinite(WP + WK), WP + WK, np.nan)
    total = np.where(total > 0, total, 0.0)
    return {
        "xM": xM,
        "xA": xA,
        "WP": WP,
        "WK": WK,
        "W_total": total,
        "modal_WP": modal_WP,
        "modal_WK": modal_WK,
        "modal_total": modal_WP + modal_WK,
        "b": coeff["b"],
        "s": coeff["s"],
        "denominator": coeff["denominator"],
    }


def sphere_total_energy_vs_xM(xM_vals, rhoM, rhoA, cM, cA, lmax):
    return sphere_energy_parts_vs_xM(xM_vals, rhoM, rhoA, cM, cA, lmax)["W_total"]


def _distance_in_samples(x, min_distance_x):
    if min_distance_x is None or min_distance_x <= 0:
        return 1
    dx = np.diff(x)
    dx = dx[np.isfinite(dx) & (dx > 0)]
    return max(1, int(np.ceil(min_distance_x / np.median(dx)))) if dx.size else 1


def detect_resonances(x, y, prominence_log10=0.35, min_distance_x=0.0, width_rel_height=0.5):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y >= 0)
    if np.count_nonzero(valid) < 5:
        return pd.DataFrame(columns=["peak_id", "index", "x_peak", "height", "log10_height", "prominence_log10", "fwhm", "Q"])
    x2, y2 = x[valid], y[valid]
    tiny = np.finfo(float).tiny
    logy = np.log10(np.maximum(y2, tiny))
    distance = _distance_in_samples(x2, min_distance_x)
    idx, props = find_peaks(logy, prominence=prominence_log10, distance=distance)
    if idx.size == 0:
        return pd.DataFrame(columns=["peak_id", "index", "x_peak", "height", "log10_height", "prominence_log10", "fwhm", "Q"])
    widths, _, left_ips, right_ips = peak_widths(y2, idx, rel_height=width_rel_height)
    sample_axis = np.arange(x2.size, dtype=float)
    left_x = np.interp(left_ips, sample_axis, x2)
    right_x = np.interp(right_ips, sample_axis, x2)
    fwhm = np.maximum(right_x - left_x, np.finfo(float).eps)
    original_indices = np.flatnonzero(valid)[idx]
    table = pd.DataFrame({
        "peak_id": np.arange(1, idx.size + 1),
        "index": original_indices,
        "x_peak": x2[idx],
        "height": y2[idx],
        "log10_height": logy[idx],
        "prominence_log10": props["prominences"],
        "fwhm": fwhm,
        "Q": x2[idx] / fwhm,
    })
    return table.sort_values("x_peak", ignore_index=True)


def modal_characterization(energy_result, peaks):
    if peaks.empty:
        return peaks.copy()
    modal = energy_result["modal_total"]
    WP, WK = energy_result["WP"], energy_result["WK"]
    rows = []
    for _, peak in peaks.iterrows():
        i = int(peak["index"])
        weights = modal[:, i]
        total = np.sum(weights)
        ell = int(np.argmax(weights)) if total > 0 else -1
        participation = weights[ell] / total if total > 0 else np.nan
        row = peak.to_dict()
        row.update({
            "dominant_l": ell,
            "dominant_fraction": participation,
            "WK_over_WP": WK[i] / WP[i] if WP[i] > 0 else np.nan,
        })
        rows.append(row)
    return pd.DataFrame(rows)


def _merge_positions(values, tolerance):
    values = np.sort(np.asarray(values, dtype=float))
    if values.size == 0:
        return values
    groups = [[values[0]]]
    for value in values[1:]:
        if value - np.mean(groups[-1]) <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return np.array([np.mean(group) for group in groups])


def real_axis_pole_candidates(energy_result, prominence_log10=0.35, min_distance_x=0.0, merge_tolerance=1.0e-3):
    x = energy_result["xM"]
    den = energy_result["denominator"]
    rows = []
    for ell in range(den.shape[0]):
        indicator = 1.0 / np.maximum(np.abs(den[ell]) ** 2, np.finfo(float).tiny)
        peaks = detect_resonances(x, indicator, prominence_log10, min_distance_x)
        for _, peak in peaks.iterrows():
            rows.append({"ell": ell, "x_candidate": peak["x_peak"], "indicator": peak["height"], "candidate_Q": peak["Q"]})
    table = pd.DataFrame(rows)
    if table.empty:
        return table, np.array([])
    merged = _merge_positions(table["x_candidate"].to_numpy(), merge_tolerance)
    return table.sort_values(["x_candidate", "ell"], ignore_index=True), merged


def complex_pole_from_seed(seed_x, ell, rhoM, rhoA, cM, cA, imag_seed=-1.0e-3, tol=1.0e-10):
    validate_material(rhoM, rhoA, cM, cA, ell)
    impedance_ratio = rhoM * cM / (rhoA * cA)
    def equations(v):
        z = v[0] + 1j * v[1]
        xA = z * cM / cA
        den = J(ell, xA) * dH(ell, z) - impedance_ratio * dJ(ell, xA) * H(ell, z)
        return [np.real(den), np.imag(den)]
    sol = root(equations, [float(seed_x), float(imag_seed)], tol=tol)
    z = sol.x[0] + 1j * sol.x[1]
    return {"ell": ell, "seed_x": seed_x, "pole_x": z, "success": bool(sol.success), "residual": float(np.linalg.norm(sol.fun))}


def phase_shift_and_wigner(energy_result, ell):
    x = energy_result["xM"]
    s = energy_result["s"][ell]
    S = 1.0 + 2.0 * s
    delta = 0.5 * np.unwrap(np.angle(S))
    tau_dimensionless = 2.0 * np.gradient(delta, x)
    return pd.DataFrame({"xM": x, "phase_shift": delta, "tau_cM_over_a": tau_dimensionless})


def _fano_profile(x, b0, b1, amplitude, q, x0, gamma):
    epsilon = 2.0 * (x - x0) / gamma
    return b0 + b1 * (x - x0) + amplitude * (q + epsilon) ** 2 / (1.0 + epsilon ** 2)


def fit_fano_peak(x, y, x_peak, fwhm=None, window_factor=3.0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if fwhm is None or not np.isfinite(fwhm) or fwhm <= 0:
        fwhm = 0.05 * max(x_peak, np.ptp(x))
    half_window = max(window_factor * fwhm, 5.0 * np.median(np.diff(x)))
    mask = np.isfinite(y) & (x >= x_peak - half_window) & (x <= x_peak + half_window)
    xx, yy = x[mask], y[mask]
    if xx.size < 8 or np.max(yy) <= 0:
        return {"success": False}
    scale = np.max(yy)
    yn = yy / scale
    y_min, y_max = float(np.min(yn)), float(np.max(yn))
    p0 = [y_min, 0.0, max(y_max - y_min, 1.0e-6), 1.0, x_peak, fwhm]
    span = max(np.ptp(xx), np.finfo(float).eps)
    bounds = ([-10.0, -100.0 / span, -20.0, -50.0, xx.min(), np.finfo(float).eps], [10.0, 100.0 / span, 20.0, 50.0, xx.max(), 2.0 * span])
    try:
        popt, _ = curve_fit(_fano_profile, xx, yn, p0=p0, bounds=bounds, maxfev=30000)
        fit = _fano_profile(xx, *popt)
        ss_res = np.sum((yn - fit) ** 2)
        ss_tot = np.sum((yn - np.mean(yn)) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        return {
            "success": True,
            "b0": popt[0] * scale,
            "b1": popt[1] * scale,
            "amplitude": popt[2] * scale,
            "q": popt[3],
            "x0": popt[4],
            "gamma": popt[5],
            "R2": r2,
            "x_fit": xx,
            "y_fit": fit * scale,
        }
    except Exception as exc:
        return {"success": False, "error": str(exc)}


def analyze_sphere_spectrum(xM_vals, rhoM, rhoA, cM, cA, lmax=10, prominence_log10=0.35, min_distance_x=0.0, width_rel_height=0.5):
    energy = sphere_energy_parts_vs_xM(xM_vals, rhoM, rhoA, cM, cA, lmax)
    peaks = detect_resonances(energy["xM"], energy["W_total"], prominence_log10, min_distance_x, width_rel_height)
    peaks = modal_characterization(energy, peaks)
    pole_table, merged_candidates = real_axis_pole_candidates(energy, prominence_log10, min_distance_x)
    rho_ratio, c_ratio = rhoA / rhoM, cA / cM
    x_ref = sphere_reference_x(rho_ratio, c_ratio)
    if not peaks.empty:
        peaks["x_over_xref"] = peaks["x_peak"] / x_ref
    summary = {
        "rho_ratio": rho_ratio,
        "c_ratio": c_ratio,
        "impedance_ratio": rho_ratio * c_ratio,
        "x_ref": x_ref,
        "n_peaks": len(peaks),
        "n_real_axis_candidates": len(merged_candidates),
        "hidden_candidates": max(len(merged_candidates) - len(peaks), 0),
        "max_energy": float(np.nanmax(energy["W_total"])),
        "first_peak_x": float(peaks.iloc[0]["x_peak"]) if not peaks.empty else np.nan,
        "first_peak_Q": float(peaks.iloc[0]["Q"]) if not peaks.empty else np.nan,
        "first_peak_l": int(peaks.iloc[0]["dominant_l"]) if not peaks.empty else -1,
    }
    return {"energy": energy, "peaks": peaks, "pole_candidates": pole_table, "merged_candidates": merged_candidates, "summary": summary}


def _empty_state_arrays(ny, nx):
    names = ["n_peaks", "n_candidates", "hidden_candidates", "first_peak_x", "first_peak_Q", "first_peak_l", "first_peak_x_over_ref", "max_log10_energy", "WK_over_WP_first"]
    data = {name: np.full((ny, nx), np.nan) for name in names}
    data["n_peaks"][:] = 0
    data["n_candidates"][:] = 0
    data["hidden_candidates"][:] = 0
    data["first_peak_l"][:] = -1
    return data


def state_map_rho_c(rho_ratios, c_ratios, xM_vals, rhoM=1024.0, cM=1522.0, lmax=10, prominence_log10=0.35, min_distance_x=0.0, progress=True):
    rho_ratios = np.asarray(rho_ratios, dtype=float)
    c_ratios = np.asarray(c_ratios, dtype=float)
    if np.any(rho_ratios <= 0) or np.any(c_ratios <= 0):
        raise ValueError("As razões de densidade e velocidade devem ser positivas.")
    data = _empty_state_arrays(c_ratios.size, rho_ratios.size)
    total_cases = rho_ratios.size * c_ratios.size
    done = 0
    for iy, c_ratio in enumerate(c_ratios):
        for ix, rho_ratio in enumerate(rho_ratios):
            result = analyze_sphere_spectrum(xM_vals, rhoM, rhoM * rho_ratio, cM, cM * c_ratio, lmax, prominence_log10, min_distance_x)
            summary, peaks = result["summary"], result["peaks"]
            data["n_peaks"][iy, ix] = summary["n_peaks"]
            data["n_candidates"][iy, ix] = summary["n_real_axis_candidates"]
            data["hidden_candidates"][iy, ix] = summary["hidden_candidates"]
            data["max_log10_energy"][iy, ix] = np.log10(max(summary["max_energy"], np.finfo(float).tiny))
            if not peaks.empty:
                first = peaks.iloc[0]
                data["first_peak_x"][iy, ix] = first["x_peak"]
                data["first_peak_Q"][iy, ix] = first["Q"]
                data["first_peak_l"][iy, ix] = first["dominant_l"]
                data["first_peak_x_over_ref"][iy, ix] = first["x_over_xref"]
                data["WK_over_WP_first"][iy, ix] = first["WK_over_WP"]
            done += 1
            if progress and (done == total_cases or done % max(1, total_cases // 20) == 0):
                print(f"Mapa rho-c: {done}/{total_cases}")
    return {"x_axis": rho_ratios, "y_axis": c_ratios, "x_label": r"$\rho_A/\rho_M$", "y_label": r"$c_A/c_M$", "data": data}


def state_map_impedance_mix(impedance_ratios, f_mix_values, xM_vals, rhoM=1024.0, cM=1522.0, lmax=10, prominence_log10=0.35, min_distance_x=0.0, progress=True):
    impedance_ratios = np.asarray(impedance_ratios, dtype=float)
    f_mix_values = np.asarray(f_mix_values, dtype=float)
    if np.any(impedance_ratios <= 0) or np.any((f_mix_values < 0) | (f_mix_values > 1)):
        raise ValueError("Use impedâncias positivas e 0 <= f_mix <= 1.")
    data = _empty_state_arrays(f_mix_values.size, impedance_ratios.size)
    total_cases = impedance_ratios.size * f_mix_values.size
    done = 0
    for iy, f_mix in enumerate(f_mix_values):
        rho_ratios, c_ratios = ratios_from_impedance(impedance_ratios, float(f_mix))
        for ix, (rho_ratio, c_ratio) in enumerate(zip(rho_ratios, c_ratios)):
            result = analyze_sphere_spectrum(xM_vals, rhoM, rhoM * rho_ratio, cM, cM * c_ratio, lmax, prominence_log10, min_distance_x)
            summary, peaks = result["summary"], result["peaks"]
            data["n_peaks"][iy, ix] = summary["n_peaks"]
            data["n_candidates"][iy, ix] = summary["n_real_axis_candidates"]
            data["hidden_candidates"][iy, ix] = summary["hidden_candidates"]
            data["max_log10_energy"][iy, ix] = np.log10(max(summary["max_energy"], np.finfo(float).tiny))
            if not peaks.empty:
                first = peaks.iloc[0]
                data["first_peak_x"][iy, ix] = first["x_peak"]
                data["first_peak_Q"][iy, ix] = first["Q"]
                data["first_peak_l"][iy, ix] = first["dominant_l"]
                data["first_peak_x_over_ref"][iy, ix] = first["x_over_xref"]
                data["WK_over_WP_first"][iy, ix] = first["WK_over_WP"]
            done += 1
            if progress and (done == total_cases or done % max(1, total_cases // 20) == 0):
                print(f"Mapa impedância-f_mix: {done}/{total_cases}")
    return {"x_axis": impedance_ratios, "y_axis": f_mix_values, "x_label": r"$Z_A/Z_M$", "y_label": r"$f_{mix}$", "data": data}


def material_atlas(materials, radius, xM_vals, rhoM=1024.0, cM=1522.0, lmax=10, prominence_log10=0.35, min_distance_x=0.0):
    if radius <= 0:
        raise ValueError("radius deve ser positivo.")
    rows = []
    details = {}
    for name, props in materials.items():
        rhoA, cA = float(props["rho"]), float(props["c"])
        result = analyze_sphere_spectrum(xM_vals, rhoM, rhoA, cM, cA, lmax, prominence_log10, min_distance_x)
        details[name] = result
        summary = result["summary"]
        frequency_ref = cM * summary["x_ref"] / (2.0 * np.pi * radius)
        frequency_first = cM * summary["first_peak_x"] / (2.0 * np.pi * radius) if np.isfinite(summary["first_peak_x"]) else np.nan
        rows.append({
            "material": name,
            "rho_A": rhoA,
            "c_A": cA,
            "rho_A/rho_M": summary["rho_ratio"],
            "c_A/c_M": summary["c_ratio"],
            "Z_A/Z_M": summary["impedance_ratio"],
            "x_ref": summary["x_ref"],
            "f_ref_Hz": frequency_ref,
            "n_peaks": summary["n_peaks"],
            "x_first": summary["first_peak_x"],
            "f_first_Hz": frequency_first,
            "Q_first": summary["first_peak_Q"],
            "l_first": summary["first_peak_l"],
            "max_energy": summary["max_energy"],
        })
    return pd.DataFrame(rows).sort_values("Z_A/Z_M", ignore_index=True), details


def match_resonances(sphere_peaks, core_shell_peaks, max_relative_gap=np.inf):
    sphere_x = np.asarray(sphere_peaks["x_peak"] if isinstance(sphere_peaks, pd.DataFrame) else sphere_peaks, dtype=float)
    core_x = np.asarray(core_shell_peaks["x_peak"] if isinstance(core_shell_peaks, pd.DataFrame) else core_shell_peaks, dtype=float)
    rows = []
    if sphere_x.size and core_x.size:
        cost = np.abs(sphere_x[:, None] - core_x[None, :]) / np.maximum(sphere_x[:, None], np.finfo(float).eps)
        i_s, i_c = linear_sum_assignment(cost)
        matched_s, matched_c = set(), set()
        for i, j in zip(i_s, i_c):
            if cost[i, j] <= max_relative_gap:
                matched_s.add(i)
                matched_c.add(j)
                rows.append({"status": "matched", "sphere_index": i + 1, "core_shell_index": j + 1, "x_sphere": sphere_x[i], "x_core_shell": core_x[j], "delta_x": core_x[j] - sphere_x[i], "x_ratio": core_x[j] / sphere_x[i]})
        for i in range(sphere_x.size):
            if i not in matched_s:
                rows.append({"status": "sphere_only", "sphere_index": i + 1, "core_shell_index": np.nan, "x_sphere": sphere_x[i], "x_core_shell": np.nan, "delta_x": np.nan, "x_ratio": np.nan})
        for j in range(core_x.size):
            if j not in matched_c:
                rows.append({"status": "core_shell_only", "sphere_index": np.nan, "core_shell_index": j + 1, "x_sphere": np.nan, "x_core_shell": core_x[j], "delta_x": np.nan, "x_ratio": np.nan})
    else:
        for i, value in enumerate(sphere_x):
            rows.append({"status": "sphere_only", "sphere_index": i + 1, "core_shell_index": np.nan, "x_sphere": value, "x_core_shell": np.nan, "delta_x": np.nan, "x_ratio": np.nan})
        for j, value in enumerate(core_x):
            rows.append({"status": "core_shell_only", "sphere_index": np.nan, "core_shell_index": j + 1, "x_sphere": np.nan, "x_core_shell": value, "delta_x": np.nan, "x_ratio": np.nan})
    return pd.DataFrame(rows)


def plot_spectrum(result, title=None, logy=True, save=None, show=True, close=True):
    energy, peaks = result["energy"], result["peaks"]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(energy["xM"], energy["W_total"], label="Total")
    ax.plot(energy["xM"], energy["WP"], label="Potential")
    ax.plot(energy["xM"], energy["WK"], label="Kinetic")
    if not peaks.empty:
        ax.scatter(peaks["x_peak"], peaks["height"], s=28, label="Resonances")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel(r"$x_M$")
    ax.set_ylabel("Normalized internal energy")
    if title:
        ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def plot_modal_structure(result, max_modes=None, save=None, show=True, close=True):
    energy = result["energy"]
    modal = energy["modal_total"]
    n_modes = modal.shape[0] if max_modes is None else min(modal.shape[0], max_modes)
    fig, ax = plt.subplots(figsize=(8, 5))
    for ell in range(n_modes):
        ax.plot(energy["xM"], modal[ell], label=fr"$\ell={ell}$")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x_M$")
    ax.set_ylabel("Modal normalized energy")
    ax.legend(ncol=2)
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def plot_pole_indicators(result, max_modes=4, save=None, show=True, close=True):
    energy = result["energy"]
    den = energy["denominator"]
    n_modes = min(den.shape[0], max_modes)
    fig, ax = plt.subplots(figsize=(8, 5))
    for ell in range(n_modes):
        indicator = 1.0 / np.maximum(np.abs(den[ell]) ** 2, np.finfo(float).tiny)
        ax.plot(energy["xM"], indicator, label=fr"$\ell={ell}$")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x_M$")
    ax.set_ylabel(r"$1/|D_\ell|^2$")
    ax.legend(ncol=2)
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def plot_fano_fit(result, peak_index=0, save=None, show=True, close=True):
    peaks = result["peaks"]
    if peaks.empty or peak_index >= len(peaks):
        return None, None, {"success": False, "error": "Nenhum pico disponível."}
    peak = peaks.iloc[peak_index]
    energy = result["energy"]
    fit = fit_fano_peak(energy["xM"], energy["W_total"], peak["x_peak"], peak["fwhm"])
    if not fit.get("success"):
        return None, None, fit
    fig, ax = plt.subplots(figsize=(8, 5))
    xfit = fit["x_fit"]
    mask = (energy["xM"] >= xfit.min()) & (energy["xM"] <= xfit.max())
    ax.plot(energy["xM"][mask], energy["W_total"][mask], label="Numerical")
    ax.plot(xfit, fit["y_fit"], "--", label="Fano fit")
    ax.set_xlabel(r"$x_M$")
    ax.set_ylabel("Normalized internal energy")
    ax.set_title(fr"Fano fit: $x_0={fit['x0']:.5g}$, $\Gamma={fit['gamma']:.3g}$, $q={fit['q']:.3g}$, $R^2={fit['R2']:.4f}$")
    ax.legend()
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax, fit


def plot_state_map(state_map, metric, log_x=True, log_y=False, save=None, show=True, close=True):
    if metric not in state_map["data"]:
        raise KeyError(f"Métrica inválida: {metric}")
    x, y = state_map["x_axis"], state_map["y_axis"]
    z = state_map["data"][metric]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    mesh = ax.pcolormesh(x, y, z, shading="auto")
    fig.colorbar(mesh, ax=ax, label=metric)
    if log_x:
        ax.set_xscale("log")
    if log_y:
        ax.set_yscale("log")
    ax.set_xlabel(state_map["x_label"])
    ax.set_ylabel(state_map["y_label"])
    ax.set_title(metric.replace("_", " "))
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def plot_reference_validation(state_map, save=None, show=True, close=True):
    x_axis, y_axis = state_map["x_axis"], state_map["y_axis"]
    first = state_map["data"]["first_peak_x"]
    if state_map["x_label"] == r"$\rho_A/\rho_M$":
        rho_grid, c_grid = np.meshgrid(x_axis, y_axis)
        x_ref = sphere_reference_x(rho_grid, c_grid)
    else:
        zeta_grid, f_grid = np.meshgrid(x_axis, y_axis)
        x_ref = np.sqrt(3.0 * zeta_grid ** (2.0 - f_grid))
    mask = np.isfinite(first) & np.isfinite(x_ref)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(x_ref[mask], first[mask], s=16)
    if np.any(mask):
        low = min(np.min(x_ref[mask]), np.min(first[mask]))
        high = max(np.max(x_ref[mask]), np.max(first[mask]))
        ax.plot([low, high], [low, high])
    ax.set_xlabel(r"$x_{M,\mathrm{ref}}$")
    ax.set_ylabel(r"Detected $x_{M,1}$")
    ax.set_xscale("log")
    ax.set_yscale("log")
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def plot_wigner(result, ell, save=None, show=True, close=True):
    data = phase_shift_and_wigner(result["energy"], ell)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(data["xM"], data["tau_cM_over_a"])
    ax.set_xlabel(r"$x_M$")
    ax.set_ylabel(r"$\tau c_M/a$")
    ax.set_title(fr"Wigner delay, $\ell={ell}$")
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax, data


def plot_material_atlas(atlas, save=None, show=True, close=True):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.scatter(atlas["rho_A/rho_M"], atlas["c_A/c_M"], s=70)
    for _, row in atlas.iterrows():
        ax.annotate(row["material"], (row["rho_A/rho_M"], row["c_A/c_M"]), xytext=(5, 5), textcoords="offset points")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\rho_A/\rho_M$")
    ax.set_ylabel(r"$c_A/c_M$")
    ax.set_title("Material atlas")
    fig.tight_layout()
    _finish_figure(fig, save, show, close)
    return fig, ax


def save_state_map_npz(state_map, filename):
    payload = {"x_axis": state_map["x_axis"], "y_axis": state_map["y_axis"]}
    payload.update(state_map["data"])
    np.savez_compressed(filename, **payload)


def run_complete_sphere_study(output_dir="resultados_esfera", config=SphereStudyConfig(), run_maps=True, show=True, save_files=False):
    out = Path(output_dir)
    if save_files:
        out.mkdir(parents=True, exist_ok=True)
    def path(name):
        return out / name if save_files else None
    xM = config.x_grid()
    materials = {"air": {"rho": 1.205, "c": 343.0}, "petroleum": {"rho": 968.75, "c": 1000.0}}
    _section("1. Atlas de materiais")
    atlas, details = material_atlas(materials, radius=1.0e-3, xM_vals=xM, rhoM=config.rhoM, cM=config.cM, lmax=config.lmax, prominence_log10=config.prominence_log10, min_distance_x=config.min_distance_x)
    if save_files:
        atlas.to_csv(path("material_atlas.csv"), index=False)
    _display_table(atlas)
    plot_material_atlas(atlas, save=path("material_atlas.png"), show=show)
    for name, result in details.items():
        _section(f"2. Espectro completo: {name}")
        summary = pd.DataFrame([result["summary"]])
        _display_table(summary)
        _display_table(result["peaks"], title=f"Picos detectados: {name}")
        _display_table(result["pole_candidates"], title=f"Candidatos do denominador: {name}", max_rows=100)
        if save_files:
            result["peaks"].to_csv(path(f"peaks_{name}.csv"), index=False)
            result["pole_candidates"].to_csv(path(f"pole_candidates_{name}.csv"), index=False)
        plot_spectrum(result, title=name, save=path(f"spectrum_{name}.png"), show=show)
        _section(f"3. Estrutura modal: {name}")
        plot_modal_structure(result, max_modes=min(8, config.lmax + 1), save=path(f"modal_{name}.png"), show=show)
        _section(f"4. Indicador de polos: {name}")
        plot_pole_indicators(result, max_modes=min(6, config.lmax + 1), save=path(f"pole_indicator_{name}.png"), show=show)
        if not result["peaks"].empty:
            _section(f"5. Ajuste de Fano: {name}")
            _, _, fano = plot_fano_fit(result, save=path(f"fano_first_peak_{name}.png"), show=show)
            if fano.get("success"):
                fano_table = pd.DataFrame([{k: v for k, v in fano.items() if np.isscalar(v)}])
                _display_table(fano_table)
                if save_files:
                    fano_table.to_csv(path(f"fano_first_peak_{name}.csv"), index=False)
            else:
                print("O ajuste de Fano não convergiu:", fano.get("error", "erro desconhecido"))
            ell = int(result["peaks"].iloc[0]["dominant_l"])
            _section(f"6. Atraso de Wigner: {name}")
            plot_wigner(result, ell, save=path(f"wigner_{name}.png"), show=show)
    maps = {}
    if run_maps:
        rho_ratios = np.geomspace(1.0e-3, 10.0, 17)
        c_ratios = np.geomspace(1.0e-2, 10.0, 17)
        _section("7. Mapa de estados em densidade e velocidade")
        rho_c_map = state_map_rho_c(rho_ratios, c_ratios, xM, config.rhoM, config.cM, config.lmax, config.prominence_log10, config.min_distance_x)
        maps["rho_c"] = rho_c_map
        if save_files:
            save_state_map_npz(rho_c_map, path("state_map_rho_c.npz"))
        for metric in ["n_peaks", "first_peak_x", "first_peak_Q", "first_peak_l", "first_peak_x_over_ref", "max_log10_energy", "hidden_candidates"]:
            plot_state_map(rho_c_map, metric, log_x=True, log_y=True, save=path(f"rho_c_{metric}.png"), show=show)
        _section("8. Validação da escala de referência no mapa rho-c")
        plot_reference_validation(rho_c_map, save=path("rho_c_reference_validation.png"), show=show)
        impedance_ratios = np.geomspace(1.0e-3, 10.0, 17)
        f_mix_values = np.linspace(0.0, 1.0, 17)
        _section("9. Mapa unificado em impedância e f_mix")
        mix_map = state_map_impedance_mix(impedance_ratios, f_mix_values, xM, config.rhoM, config.cM, config.lmax, config.prominence_log10, config.min_distance_x)
        maps["impedance_mix"] = mix_map
        if save_files:
            save_state_map_npz(mix_map, path("state_map_impedance_mix.npz"))
        for metric in ["n_peaks", "first_peak_x", "first_peak_Q", "first_peak_l", "first_peak_x_over_ref", "max_log10_energy", "hidden_candidates"]:
            plot_state_map(mix_map, metric, log_x=True, log_y=False, save=path(f"impedance_mix_{metric}.png"), show=show)
        _section("10. Validação da escala de referência no mapa impedância-f_mix")
        plot_reference_validation(mix_map, save=path("impedance_mix_reference_validation.png"), show=show)
    if save_files:
        print(f"Arquivos também salvos em: {out.resolve()}")
    return {"output_dir": str(out) if save_files else None, "atlas": atlas, "details": details, "maps": maps}


if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    config = SphereStudyConfig(xM_min=1.0e-4, xM_max=1.0, n_x=1800, lmax=10, prominence_log10=0.35)
    results = run_complete_sphere_study(config=config, run_maps=True, show=True, save_files=False)
